# 2.2 — Feature Engineering Pipeline

Computes per-request features from raw assessment data and validates quality.

**Note:** Session-level features (`ip_*`) are computed by the pipeline but only included in the baseline model (2.3) via causal windowing. TLS-granularity session features (`tls_*`) were removed entirely because TLS fingerprints act as perfect label proxies in this synthetic dataset — labels are partially assigned by TLS fingerprint, creating a circular dependency. See `docs/2.2-feature-engineering-decisions.md` for details.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
import numpy as np

from src.label_joining import join_labels
from src.features import compute_per_request_features
from src.pipeline import compute_sample_weights
from src.model import get_feature_columns

requests = pd.read_csv('data/http_requests.csv', parse_dates=['timestamp'])
headers = pd.read_csv('data/request_headers.csv')
labels = pd.read_csv('data/incident_labels.csv', parse_dates=['active_from', 'active_until', 'labeled_at'])

df = join_labels(requests, labels)
df = compute_per_request_features(df, headers)
df['sample_weight'] = compute_sample_weights(df)

print(f'Shape: {df.shape}')
print(f'Malicious: {df.is_malicious.sum()} ({df.is_malicious.mean()*100:.2f}%)')

Shape: (50000, 32)
Malicious: 592 (1.18%)


In [2]:
feature_cols = get_feature_columns(df)
print(f'Total feature columns: {len(feature_cols)}')
print(f'Features: {feature_cols}')

Total feature columns: 15
Features: ['path_depth', 'path_length', 'path_entropy', 'path_has_params', 'ua_length', 'ua_entropy', 'ua_is_browser', 'ua_is_bot_library', 'hour_of_day', 'is_sensitive_endpoint', 'header_count', 'has_accept_language', 'has_referer', 'has_cookie', 'has_authorization']


In [3]:
df[feature_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
path_depth,50000.0,3.467580,0.782356,1.000000,3.000000,4.000000,4.000000,7.000000
path_length,50000.0,18.759440,4.355464,6.000000,17.000000,18.000000,21.000000,33.000000
path_entropy,50000.0,3.465644,0.301371,2.521641,3.403989,3.503258,3.588354,4.088779
path_has_params,50000.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ua_length,50000.0,98.129060,37.617772,10.000000,80.000000,117.000000,118.000000,135.000000
ua_entropy,50000.0,4.849713,0.522324,3.121928,4.752567,5.086734,5.110059,5.182870
ua_is_browser,50000.0,0.845120,0.361794,0.000000,1.000000,1.000000,1.000000,1.000000
ua_is_bot_library,50000.0,0.080120,0.271482,0.000000,0.000000,0.000000,0.000000,1.000000
hour_of_day,50000.0,14.127620,5.235578,0.000000,11.000000,15.000000,18.000000,23.000000
is_sensitive_endpoint,50000.0,0.294460,0.455804,0.000000,0.000000,0.000000,1.000000,1.000000


In [4]:
numeric_features = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
comparison = df.groupby('is_malicious')[numeric_features].mean().T
comparison.columns = ['benign_mean', 'malicious_mean']
comparison['ratio'] = comparison['malicious_mean'] / comparison['benign_mean'].replace(0, float('nan'))
comparison.sort_values('ratio', ascending=False)

,benign_mean,malicious_mean,ratio
ua_is_bot_library,0.073632,0.621622,8.442298
is_sensitive_endpoint,0.292827,0.430743,1.470982
path_length,18.753360,19.266892,1.027383
path_entropy,3.465019,3.517790,1.015230
path_depth,3.467151,3.503378,1.010449
ua_entropy,4.855794,4.342163,0.894223
hour_of_day,14.173211,10.322635,0.728320
ua_length,98.640038,55.483108,0.562481
header_count,2.944604,1.608108,0.546120
ua_is_browser,0.850712,0.378378,0.444778


In [5]:
import numpy as np

print('NaN counts:')
nan_counts = df[feature_cols].isna().sum()
print(nan_counts[nan_counts > 0] if nan_counts.sum() > 0 else 'None')

print(f'\nInf counts:')
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(df[numeric_cols]).sum()
print(inf_counts[inf_counts > 0] if inf_counts.sum() > 0 else 'None')

NaN counts:
None

Inf counts:
None


In [6]:
# No longer saving training_data.csv — baseline model (2.3) loads raw data directly
# to avoid leakage from pre-computed features.
print("Feature engineering complete. Baseline model loads raw data and computes features inline.")

Feature engineering complete. Baseline model loads raw data and computes features inline.
